## Donut 🍩, Document understanding transformer

![model image](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/donut_architecture.jpg)

[paper: OCR-free Document Understanding Transformer](https://huggingface.co/papers/2111.15664)

[clovaai/donut Github](https://github.com/clovaai/donut?tab=readme-ov-file)

[donut-base model on huggingface](https://huggingface.co/naver-clova-ix/donut-base/)


### TEXT FORMAT

**during training:**

- **INPUT:**
  - Image embedding
  - task_start_token + bos_token + json2token(ground_truth_json) + eos_token

**during inference:**

- **INPUT:**
  - Image embedding
  - task_start_token

- **OUTPUT:**
  - ground_truth_json

### ⚙️ RTX 5090 / 로컬(conda `donut_vml`) 적응판

원본 Colab 노트북(`source_colab.ipynb`)을 로컬 RTX 5090 환경에 맞게 수정한 버전입니다.

**주요 변경**
- `fp16=True` → **`bf16=True`** (Donut은 fp16에서 수치 불안정; 5090은 bf16 지원)
- `/content/...` Colab 경로 → **로컬 경로**, `hftuner` 라이브러리를 폴더에 클론 후 `sys.path` 추가
- `huggingface_hub.login()` / `push_to_hub` / 모델카드 푸시 **제거** (로컬 저장만)
- `report_to="tensorboard"` → `"none"`, eval batch 8→4 (고해상도 generate 메모리)
- 추론 dtype fp16 → bf16, 평가 `test.py` 경로/모델 로컬화

> ✅ **검증됨(transformers 5.12.1)**: import · `DonutModel.from_pretrained`(200.3M) · cell 25 의
> MBart 내부 경로까지 정상 동작. 단 학습/generate 전체는 미검증 — 5.x API 차로 막히면
> 첫 코드 셀의 `%pip install "transformers==4.56.1"` 주석을 푸세요(가급적 별도 env).
> RVL-CDIP 데이터셋은 첫 실행 시 HF에서 대용량 다운로드됩니다.

In [3]:
# ── [로컬/VS Code Jupyter] widget 렌더러 오류 방지 + 토큰 경고 안내 ─────────────
# "Could not render content for application/vnd.jupyter.widget-view+json" 는
# datasets/HF 의 tqdm.auto 진행바가 ipywidget 을 그리려다 실패하는 *표시 문제*(에러 아님).
# 아래로 텍스트 진행바로 강제해 없앱니다. (도면 학습 노트북과 동일한 픽스)
import os
os.environ['TQDM_NOTEBOOK'] = 'false'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# HF_TOKEN 미설정 경고는 공개 모델/데이터(RVL-CDIP·donut-classification-turbo)엔 무해합니다.
# 비공개 자원/다운로드 속도·레이트리밋 개선이 필요하면 토큰 설정(선택):
#   os.environ['HF_TOKEN'] = 'hf_...'  # 따옴표 안에 본인 토큰
print('widget/token env 설정 완료')

widget/token env 설정 완료


In [4]:
# [로컬/RTX5090] donut_vml conda 환경엔 transformers가 이미 설치돼 있습니다.
# 원본은 transformers==4.56.1 핀이며 hftuner 라이브러리가 이 버전대를 가정합니다.
# DonutModel 로드/학습이 버전 차로 깨지면 아래 줄의 주석을 풀어 맞추세요.
# %pip install "transformers==4.56.1" --quiet
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

transformers 5.12.1 | torch 2.11.0+cu128 | cuda True


In [5]:
# [로컬] hftuner(clovaai-donut) 라이브러리를 이 노트북 폴더에 클론하고 import 경로 추가.
import os, sys
_here   = os.path.abspath(os.getcwd())
HFTUNER = os.path.join(_here, "hftuner")
if not os.path.isdir(HFTUNER):
    !git clone https://github.com/hftuner/clovaai-donut.git "{HFTUNER}"
if _here not in sys.path:
    sys.path.insert(0, _here)   # 부모 디렉터리를 path에 → `import hftuner` 가능
print("hftuner ready:", os.path.isdir(HFTUNER))

hftuner ready: True


In [6]:
import os
import json
from datasets import load_dataset

dataset = load_dataset('hf-tuner/rvl-cdip-document-classification')

class_names = dataset['train'].features['label'].names
id2label = {i: c for i, c in enumerate(class_names)}

# add ground truth
def add_gt(example):
  class_name = id2label[example['label']]
  example["ground_truth"] = json.dumps({
      "gt_parse":{
          "class": class_name # str
      }
  })
  return example

dataset = dataset.map(add_gt)

Map: 100%|██████████| 992/992 [00:00<00:00, 15449.44 examples/s]


In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'ground_truth'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['image', 'label', 'ground_truth'],
        num_rows: 992
    })
})

In [6]:
dataset_info = dataset['train'].features
dataset_info

{'image': Image(mode=None, decode=True),
 'label': ClassLabel(names=['letter', 'form', 'email', 'handwritten', 'advertisement', 'scientific report', 'scientific publication', 'specification', 'file folder', 'news article', 'budget', 'invoice', 'presentation', 'questionnaire', 'resume', 'memo']),
 'ground_truth': Value('string')}

In [8]:
ignore_id = -100
task_start_token = "<classification>"
new_special_tokens = []

# add tokens from class_names
for class_name in class_names:
  class_name = class_name.replace(" ","_")
  new_special_tokens.append(class_name)

new_special_tokens.extend([task_start_token])
new_special_tokens

['letter',
 'form',
 'email',
 'handwritten',
 'advertisement',
 'scientific_report',
 'scientific_publication',
 'specification',
 'file_folder',
 'news_article',
 'budget',
 'invoice',
 'presentation',
 'questionnaire',
 'resume',
 'memo',
 '<classification>']

In [9]:
from hftuner.donut import DataProcessor

data_processor = DataProcessor()
json2token = data_processor.json2token

example_document_classification_gt = {"class" : "scientific_report"} #gt_parse

json2token(example_document_classification_gt)

'<s_class>scientific_report</s_class>'

## special tokens from dataset

In [10]:
def clean_docs_for_donut(sample):
    gt = json.loads(sample["ground_truth"])
    parsed_text = gt['gt_parse'] if 'gt_parse' in gt else gt['gt_parses']
    text = json2token(parsed_text, update_special_tokens_for_json_key=True) # adds a new token for each key
    return {"text": text}

columns_to_remove = [ c for c in dataset['train'].column_names if c not in ['image', 'ground_truth', 'text']]

data_processor.clear_new_special_tokens()
proc_dataset = dataset.map(clean_docs_for_donut, remove_columns=columns_to_remove)
new_key_tokens = data_processor.get_new_special_tokens()

# print(f"Sample:\n{proc_dataset['train'][3]['text']}")
print(f"Sample:\n{proc_dataset['train'][4]}")
new_key_tokens
# bos_token and eos_token will be added by data collator function

Map: 100%|██████████| 992/992 [00:00<00:00, 15865.22 examples/s]

Sample:
{'image': <PIL.Image.Image image mode=L size=794x1000 at 0x73DC1B78FAC0>, 'ground_truth': '{"gt_parse": {"class": "memo"}}', 'text': '<s_class>memo</s_class>'}


['</s_class>', '<s_class>']

## load model and processor

In [11]:
import torch
from transformers import DonutProcessor, VisionEncoderDecoderConfig
from hftuner.donut import DonutModel

# ckpt = 'naver-clova-ix/donut-base'
ckpt = 'hf-tuner/donut-classification-turbo'
processor = DonutProcessor.from_pretrained(ckpt, use_fast=True)
model = DonutModel.from_pretrained(ckpt)

len(processor.tokenizer)

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Loading weights: 100%|██████████| 483/483 [00:00<00:00, 21093.80it/s]


57542

### Configure processor and model for finetuning

In [12]:
new_special_tokens = list(set(new_special_tokens + new_key_tokens))
print(f"New special tokens:  {new_special_tokens}")

# add new special tokens to tokenizer
print(f"Adding {len(new_special_tokens)} special tokens")
processor.tokenizer.add_special_tokens({"additional_special_tokens": new_special_tokens})
print(f"New tokenizer length: {len(processor.tokenizer)}")


New special tokens:  ['memo', 'form', '<s_class>', 'file_folder', 'letter', 'news_article', 'questionnaire', 'handwritten', 'resume', 'email', 'specification', 'budget', '</s_class>', '<classification>', 'advertisement', 'scientific_publication', 'scientific_report', 'presentation', 'invoice']
Adding 19 special tokens
New tokenizer length: 57542


In [13]:
# to find max_token to output during training
def add_token_count(example):
  input_ids = processor.tokenizer(
      example['text'],
      add_special_tokens=True,
      # return_tensors="pt",
  ).input_ids
  example['tokens_count'] = len(input_ids)
  return example

proc_dataset = proc_dataset.map(add_token_count)
max_length = max(proc_dataset['train']['tokens_count'])
print('maximum length of example:', max_length)


Map: 100%|██████████| 992/992 [00:00<00:00, 11845.28 examples/s]


maximum length of example: 6


In [14]:
max_length = 8
processor_image_size = [960, 1280] # (width, height)


## UPDATE: processor
processor.image_processor.size['width'] = processor_image_size[0]
processor.image_processor.size['height'] = processor_image_size[1]
processor.image_processor.do_align_long_axis = False # don't rotate image if height is greater than width

## UPDATE: model

# generation config
model.config.decoder.max_length = max_length
model.config.pad_token_id = processor.tokenizer.pad_token_id
# ⚠️ IMPORTANT: set decoder_start_token
task_token_id = processor.tokenizer.encode(task_start_token, add_special_tokens=False)[0]
model.config.decoder_start_token_id = task_token_id

# Resize embedding layer to match vocabulary size
new_emb = model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Adjust our image size and output sequence lengths
model.config.encoder.image_size = processor_image_size[::-1] # (height, width)

print(f"New token embedding size: {new_emb}")

New token embedding size: MBartScaledWordEmbedding(57542, 1024, padding_idx=1)


More about : [decoder_start_token_id](https://github.com/huggingface/transformers/blob/4df2529d79d75f44e70396df5888a32ffa02d61e/src/transformers/models/vision_encoder_decoder/modeling_vision_encoder_decoder.py#L331)

## Data Collator

In [15]:
from torchvision.transforms import Compose, ColorJitter
jitter = Compose(
    [
         ColorJitter(brightness=0.45, contrast=0.35, saturation=0.25, hue=0.4),
    ]
)

In [16]:
def prepare_data(examples):
    images = [e['image'].convert("RGB") for e in examples]
    # images = [ jitter(img) for img in images] # data_augmentation
    texts = [e['text'] for e in examples]
    # create tensor from image
    pixel_values = processor(
        images,
        return_tensors="pt"
        ).pixel_values
    # tokenize text
    input_ids = processor.tokenizer(
        texts,
        add_special_tokens=True, # add <s> and </s>
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).input_ids
    labels = input_ids.clone()
    # ignore pad tokens when calculating `loss`
    labels[labels == processor.tokenizer.pad_token_id] = ignore_id
    return {"pixel_values": pixel_values, "labels": labels}

# TEST
batch= prepare_data([
    proc_dataset['train'][5],
])
batch.keys(), batch['pixel_values'].shape, batch['labels'].shape

(dict_keys(['pixel_values', 'labels']),
 torch.Size([1, 3, 1280, 960]),
 torch.Size([1, 8]))

## ⚙️ Efficient fine tuning

In [17]:
f"number of encoder parameters: {(model.encoder.num_parameters() / 1e6):.2f}M"

'number of encoder parameters: 74.18M'

In [18]:
for p in model.encoder.parameters():
  p.requires_grad = False

In [19]:
# trainer.get_num_trainable_parameters() / 1e6 # freezed decoder

In [20]:
import torch.nn.functional as F

def resize_bart_abs_pos_emb(weight: torch.Tensor, max_length: int) -> torch.Tensor:
  """
  Resize position embeddings
  Truncate if sequence length of Bart backbone is greater than given max_length,
  else interpolate to max_length
  """
  if weight.shape[0] > max_length:
      weight = weight[:max_length, ...]
  else:
      weight = (
          F.interpolate(
              weight.permute(1, 0).unsqueeze(0),
              size=max_length,
              mode="linear",
              align_corners=True,
          )
          .squeeze(0)
          .permute(1, 0)
      )
  return weight

In [21]:
model.decoder.model.decoder.embed_positions.weight = torch.nn.Parameter(resize_bart_abs_pos_emb(model.decoder.model.decoder.embed_positions.weight, 10 ))
model.config.decoder.max_position_embeddings = 8

In [22]:
# model.decoder.model.decoder
model.decoder.model.decoder.embed_positions.weight.shape

torch.Size([10, 1024])

In [23]:
# trainer.get_num_trainable_parameters() / 1e6 # reduced decoder.embed_positions

## ⚙️ Train

In [24]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

save_model_name = "donut-classification-turbo"
# 사전학습 lr: [1e-5, 1e-4] — 그 이하/비슷하게 사용

# [로컬] huggingface_hub.login() 제거 — Hub 푸시를 끄므로 인증 불필요.
training_args = Seq2SeqTrainingArguments(
    output_dir=save_model_name,
    num_train_epochs=3,
    logging_steps=100,
    learning_rate=1e-5,
    per_device_train_batch_size=2,    # RTX 5090(32GB): GPU가 비면 4~8까지 상향 가능
    per_device_eval_batch_size=4,     # 960x1280 + generate는 메모리 큼 → 보수적
    weight_decay=0.01,                # 암기(과적합) 억제
    bf16=True,        # [RTX 5090/Donut] fp16은 Donut에서 수치 불안정 → bf16 사용
    fp16=False,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="epoch",
    save_total_limit=1,
    predict_with_generate=True,
    remove_unused_columns=False,
    report_to="none",   # 원본 "tensorboard" → 의존성 줄이려 none (원하면 되돌리기)
    push_to_hub=False,  # [로컬] HF Hub 푸시 비활성 (원본은 hf-tuner 조직으로 push)
)

# Create Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=prepare_data,
    train_dataset=proc_dataset["train"],
    eval_dataset=proc_dataset["test"],
)

In [25]:
trainer.get_num_trainable_parameters() / 1e6

126.124032

In [26]:
trainer.get_num_trainable_parameters() / 1e6

126.124032

In [27]:
# Start training
trainer.train()

Step,Training Loss,Validation Loss
2000,0.042607,0.071837
4000,0.071948,0.045039
6000,0.083035,0.039615
8000,0.058415,0.030822
10000,0.047710,0.025726
12000,0.046448,0.023595


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.58it/s]


TrainOutput(global_step=12000, training_loss=0.05196305518349012, metrics={'train_runtime': 808.5427, 'train_samples_per_second': 29.683, 'train_steps_per_second': 14.842, 'total_flos': 7.50458796244992e+19, 'train_loss': 0.05196305518349012, 'epoch': 3.0})

In [28]:
# [로컬] 로컬 저장만 — HF Hub 푸시/모델카드 생성 제거
trainer.save_model(save_model_name)
processor.save_pretrained(save_model_name)
print("saved ->", os.path.abspath(save_model_name))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.37it/s]

saved -> /home/jhkim/donut_vml/donut_finetune_classification/donut-classification-turbo


## 🎯 Evaluate

In [ ]:
%pip install evaluate --quiet

In [ ]:
# [로컬] hftuner/scripts/test.py 로 평가 — 로컬에 저장한 모델 사용
import os
_model_path = os.path.abspath(save_model_name)
%cd hftuner/scripts
!python test.py --pretrained_model_name_or_path "{_model_path}" \
  --dataset_name_or_path "hf-tuner/rvl-cdip-document-classification" \
  --task_start_token "<classification>" \
  --task_type "classification" \
  --eval_batch_size 8 --save_predictions
%cd -

## 🚀 Inference

In [29]:
import re
import torch
from transformers import DonutProcessor, VisionEncoderDecoderConfig
from hftuner.donut import DonutModel

ckpt = save_model_name   # [로컬] 위에서 학습·저장한 모델 디렉터리
# ckpt = "hf-tuner/donut-base-finetuned-rvl-cdip"  # 또는 허브 모델

config    = VisionEncoderDecoderConfig.from_pretrained(ckpt)
processor = DonutProcessor.from_pretrained(ckpt)
device    = "cuda" if torch.cuda.is_available() else "cpu"
# [RTX 5090/Donut] fp16 대신 bf16 (Donut fp16 불안정)
config.dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = DonutModel.from_pretrained(ckpt, config=config)
model.to(device)

Loading weights: 100%|██████████| 483/483 [00:00<00:00, 8203.31it/s]


DonutModel(
  (encoder): DonutSwinModel(
    (embeddings): DonutSwinEmbeddings(
      (patch_embeddings): DonutSwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DonutSwinEncoder(
      (layers): ModuleList(
        (0): DonutSwinStage(
          (blocks): ModuleList(
            (0): DonutSwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): DonutSwinAttention(
                (self): DonutSwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )
              

In [30]:
from datasets import load_dataset

dataset = load_dataset('hf-tuner/rvl-cdip-document-classification')

In [32]:
# inference (generation)
task_start_token = "<classification>"
sample = dataset['test'][21]

# [RTX5090/bf16] 모델이 bf16 이므로 입력 pixel_values 도 같은 dtype 으로 맞춰야 함
# (안 맞추면 "Input type (float) and bias type (BFloat16) should be the same" 에러)
pixel_values = processor(sample['image'], return_tensors="pt").pixel_values.to(device, model.dtype)

decoder_input_ids = processor.tokenizer(task_start_token, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(pixel_values,
                               decoder_input_ids=decoder_input_ids,
                               max_length=16,
                               bad_words_ids=[[processor.tokenizer.unk_token_id]]
                               )

print(f'generated tokens count: {len(generated_ids[0])}')
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
print(generated_text)
processor.token2json(generated_text)

generated tokens count: 6
<classification><s><s_class>letter</s_class></s>


{'class': 'letter'}

In [ ]:
print(sample['label'])
# sample['image'].resize((500,800))

0


: 